# 댓글 감정 분석
## Week06 구조로 우리 CSV 학습하기

본문은 translated_ko, GT는 숫자 label입니다. 0=부정, 1=긍정, 2=중립, 3=혼합, 4=판단 불가의 5개 클래스를 모두 학습합니다. data/comments_261010.csv는 두 원본 CSV의 201행·30컬럼을 보존한 병합 파일입니다. 댓글 없는 메모 1행은 전처리에서 제외합니다. label_name은 정답 기준으로 쓰지 않으며, 숫자 label과의 불일치 수는 전처리 요약에서 확인할 수 있습니다.

## 1 준비와 경로 확인
현재 노트북의 폴더를 기준으로 같은 폴더의 Python 파일과 하위 data/outputs를 참조합니다. VS Code가 제공한 노트북 경로를 우선 사용하고, 일반 Jupyter에서는 커널 작업 폴더를 사용합니다. 작업 폴더가 상위 폴더라면 프로젝트 이름을 고정하지 않고 그 아래의 노트북 폴더를 찾습니다. 후보가 여러 개면 임의로 선택하지 않습니다. 커널은 이 폴더의 .venv를 선택하세요.

In [12]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

# 노트북에는 __file__이 없으므로 편집기 경로 또는 커널 작업 폴더를 사용합니다.
notebook_file = globals().get("__vsc_ipynb_file__")
LAB = Path(notebook_file).expanduser().resolve().parent if notebook_file else Path.cwd().resolve()
def is_lab_directory(folder):
    return all((folder / name).is_file() for name in ["train_sentiment.py", "train_lstm.py", "label_schema.json"])
if not is_lab_directory(LAB):
    if notebook_file:
        raise FileNotFoundError(f"노트북과 같은 폴더에 학습 코드가 없습니다: {LAB}")
    candidates = []
    for root, dirs, files in os.walk(LAB):
        dirs[:] = [name for name in dirs if not name.startswith(".") and name not in {"__pycache__", "node_modules", "outputs", "venv"}]
        folder = Path(root).resolve()
        if any(name.endswith(".ipynb") for name in files) and is_lab_directory(folder):
            candidates.append(folder)
    if len(candidates) != 1:
        raise FileNotFoundError("노트북 폴더를 하나로 찾을 수 없습니다. 커널 작업 폴더를 현재 노트북의 폴더로 변경한 후 첫 셀부터 실행하세요.")
    LAB = candidates[0]
# 다른 복사본의 모듈이 커널에 남아 있으면 현재 폴더에서 다시 가져옵니다.
for name in ["train_sentiment", "train_lstm"]:
    cached = sys.modules.get(name)
    if cached is not None and Path(cached.__file__).resolve().parent != LAB:
        del sys.modules[name]
sys.path.insert(0, str(LAB))
from train_sentiment import (load_data, split_data, make_model, compare_models,
    prediction_table, feature_weights, run_experiment, LABELS, LABEL_NAMES, CANDIDATES)
CSV_PATH = LAB / "data" / "comments_261010.csv"
MODEL_NAMES = list(CANDIDATES)
INCLUDE_LSTM = True  # requirements-lstm.txt 설치 후 True
LSTM_EPOCHS = 20
print("데이터:", CSV_PATH.resolve())

데이터: C:\Users\ktc system\나\3-2\culture_NLP\sub_assignment\261008\CNT3054-main\emotion_analysis\data\comments_261010.csv


## 2 데이터와 GT 확인
0~4의 다섯 라벨을 사용합니다. 판단 불가는 빈 라벨과 다릅니다. parent_id/comment_type이 있으면 답글을 제외하고, 댓글·GT·ID가 없는 메모 행도 제외합니다. label_name/human_label/is_trainable은 GT를 바꾸지 않습니다.

In [13]:
data, data_info = load_data(CSV_PATH)
display(data[["review_id", "group_id", "translated_ko", "text", "label"]].head(8))
display(data["label"].map(LABEL_NAMES).value_counts())
print(data_info)

,review_id,group_id,translated_ko,text,label
0,UgztJ4Mpla2z7ZMJ-f54AaABAg,GROUP000001,병음 행위,병음 행위,0
1,Ugwddt76HB44FA2uTzZ4AaABAg,GROUP000002,새 여성 아티스트상을 수상해야 합니다. ❣️,새 여성 아티스트상을 수상해야 합니다. ❣️,1
2,UgxG3sZ4uMCmDb3qobN4AaABAg,GROUP000003,팬도라스 상자를 열는 것에 대해 노래하는 예술가들의 어두운 코미디는 내 마음에 사라...,팬도라스 상자를 열는 것에 대해 노래하는 예술가들의 어두운 코미디는 내 마음에 사라...,4
3,UgyzSNqrVmj54G1f4Bt4AaABAg,GROUP000004,한국어 댓이 없어…! 누군데 메이브가!!!!!!! 가상현실이 에스파 생각만 나는데,한국어 댓이 없어...! 누군데 메이브가!!!!!!! 가상현실이 에스파 생각만 나는데,2
4,UgyGINaFv94fj6GeR8p4AaABAg,GROUP000005,오그 노래 MAVE-PANDORA 이 AESPA보다 훨씬 더 좋은 노래,오그 노래 MAVE-PANDORA 이 AESPA보다 훨씬 더 좋은 노래,1
5,UgwG-wWZce1CAJ0DsRV4AaABAg,GROUP000006,아주 이상하지만 러시아어 없습니다 노래는 인기가 있지만,아주 이상하지만 러시아어 없습니다 노래는 인기가 있지만,2
6,UgzZ5sF1-UEZJhuFTMR4AaABAg,GROUP000007,"TBH, 저는 이 CGI 아이돌보다 그들의 삶과 시간에 연습하는 진짜 아이돌들을 존...","TBH, 저는 이 CGI 아이돌보다 그들의 삶과 시간에 연습하는 진짜 아이돌들을 존...",0
7,UgzvIkBWPzLS66bjkCd4AaABAg,GROUP000008,러시아인 누구죠?,러시아인 누구죠?,2


label
긍정       92
부정       57
중립       37
혼합        9
판단 불가     5
Name: count, dtype: int64

{'input_rows': 201, 'annotation_rows_removed': 1, 'replies_removed': 0, 'top_level_rows': 200, 'deduplicated_rows': 200, 'duplicates_removed': 0, 'groups': 200, 'source_types': ['user_provided_comments'], 'text_column': 'translated_ko', 'target_column': 'label', 'label_names': {0: '부정', 1: '긍정', 2: '중립', 3: '혼합', 4: '판단 불가'}, 'label_name_mismatches': 62, 'gt_policy': 'numeric label is authoritative; label_name/human_label/is_trainable are not GT', 'group_method': 'connected components of text, provided group_id and author_channel_id', 'preprocessing': 'NFKC, HTML unescape, whitespace; URL/email/phone/handle placeholders; punctuation and emoji preserved'}


## 3 먼저 분리하기
seed 42, 그룹과 클래스 분포를 고려한 약 60/20/20입니다. 작성자/중복/지정 그룹은 분할을 넘지 않습니다. 모든 후보와 LSTM이 동일한 분할을 씁니다.

In [14]:
splits = split_data(data)
train, validation, test = (splits[k] for k in ["train", "validation", "test"])
display(pd.DataFrame({k: v["label"].value_counts().reindex(LABELS, fill_value=0)
                      for k, v in splits.items()}))
print({k: len(v) for k, v in splits.items()})

,train,validation,test
label,,,
0,34,11,12
1,55,19,18
2,22,7,8
3,6,2,1
4,3,1,1


{'train': 120, 'validation': 40, 'test': 40}


## 4 핵심 모델 TF IDF와 분류기
수업처럼 Pipeline의 fit은 train에서만 호출합니다. 한 글자와 이모지·문장부호를 보존하도록 토큰 패턴을 확장했습니다.

In [15]:
unigram = make_model((1, 1))
unigram.fit(train["text"], train["label"])
print(unigram)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(token_pattern='(?u)\\b\\w+\\b|[^\\w\\s]')),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


## 5 문장이 숫자가 된 모습
TF-IDF 값은 감정 점수나 확률이 아닙니다. 출력은 학습 문장 일부만 확인합니다.

In [16]:
vectorizer = unigram.named_steps["tfidf"]
X_train = vectorizer.transform(train["text"])
features = vectorizer.get_feature_names_out()
print("학습 행렬:", X_train.shape)
display(pd.DataFrame(X_train[:5, :min(12, len(features))].toarray(),
                     columns=features[:12]))

학습 행렬: (120, 1826)


,!,"""",',(,),",",-,.,/,0,00,1
0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.127028,0.0,0.0,0.0,0.0
1,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.095626,0.0,0.0,0.0,0.0
2,0.844618,0.0,0.0,0.0,0.0,0.000000,0.0,0.178397,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.0,0.0,0.0,0.109525,0.0,0.000000,0.0,0.0,0.0,0.0


## 6 기준선과 후보 비교
Dummy와 LR 두 설정, LinearSVC, MultinomialNB, XGBoost를 비교합니다. INCLUDE_LSTM=True면 1종을 추가합니다. 선택은 validation Macro-F1, 동점은 사전 정의한 후보 순서입니다.

In [17]:
models, baseline, scores, selected_name = compare_models(
    train, validation, model_names=MODEL_NAMES, include_lstm=INCLUDE_LSTM,
    lstm_epochs=LSTM_EPOCHS)
display(scores.round(4))
model = models[selected_name]
print("선택:", selected_name, "/ 검증 재학습 없이 이 모델을 유지합니다.")

,model,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,fit_seconds
0,majority_baseline,0.475,0.0950,0.2000,0.1288,0.3059,0.0007
1,unigram,0.475,0.0950,0.2000,0.1288,0.3059,0.0130
2,unigram_bigram,0.475,0.0950,0.2000,0.1288,0.3059,0.0459
3,linear_svm,0.500,0.2027,0.2182,0.1665,0.3646,0.0085
4,multinomial_nb,0.475,0.0950,0.2000,0.1288,0.3059,0.0077
5,xgboost,0.525,0.2738,0.2927,0.2784,0.4946,0.6649
6,lstm,0.475,0.0950,0.2000,0.1288,0.3059,4.0174


선택: xgboost / 검증 재학습 없이 이 모델을 유지합니다.


## 7 혼동행렬과 클래스별 지표
순서는 [0,1,2,3,4], 행=실제, 열=예측입니다. 이진 FP/FN 해석을 다섯 클래스에 그대로 적용하지 않습니다.

In [18]:
from sklearn.metrics import confusion_matrix, classification_report
validation_result = prediction_table(model, validation)
cm = confusion_matrix(validation_result["label"], validation_result["prediction"], labels=LABELS)
names = [f"{k} {LABEL_NAMES[k]}" for k in LABELS]
display(pd.DataFrame(cm, index=names, columns=names))
print(classification_report(validation_result["label"], validation_result["prediction"],
      labels=LABELS, target_names=names, zero_division=0, digits=4))

,0 부정,1 긍정,2 중립,3 혼합,4 판단 불가
0 부정,7,3,1,0,0
1 긍정,3,13,2,1,0
2 중립,2,4,1,0,0
3 혼합,1,1,0,0,0
4 판단 불가,1,0,0,0,0


              precision    recall  f1-score   support

        0 부정     0.5000    0.6364    0.5600        11
        1 긍정     0.6190    0.6842    0.6500        19
        2 중립     0.2500    0.1429    0.1818         7
        3 혼합     0.0000    0.0000    0.0000         2
     4 판단 불가     0.0000    0.0000    0.0000         1

    accuracy                         0.5250        40
   macro avg     0.2738    0.2927    0.2784        40
weighted avg     0.4753    0.5250    0.4946        40



## 8 오분류 원문 읽기
오류는 실제→예측 방향입니다. p_0~p_4는 클래스 확률, SVM score_0~score_4는 분류 점수입니다. 미등록 표현이나 낮은 확률을 자동 4로 바꾸지 않습니다.

In [19]:
errors = validation_result.loc[~validation_result["correct"]]
display(errors.head(10))
print("검증 오류 수:", len(errors))

,review_id,group_id,video_id,translated_ko,text,label,prediction,label_name,prediction_name,p_0,p_1,p_2,p_3,p_4,correct,error_type,known_features
5,UgxtGO9bqoTjvxrSxC14AaABAg,GROUP000029,1wGOHbcQKIc,", , , , , , , , , , , , , , , , , , , , , , , ...",", , , , , , , , , , , , , , , , , , , , , , , ...",4,0,판단 불가,부정,0.485395,0.339266,0.147869,0.008302,0.019169,False,4->0,1
6,Ugx440EkwgHm4R-cuDx4AaABAg,GROUP000035,1wGOHbcQKIc,와우,와우,2,0,중립,부정,0.424486,0.296694,0.264147,0.007120,0.007554,False,2->0,0
8,UgybPpI-fu1Tg4aUfCt4AaABAg,GROUP000051,1wGOHbcQKIc,만약 그룹이 이 음악과 MV를 활용한다면 얼마나 큰 폭력을 일으킬 수 있을지 상상해...,만약 그룹이 이 음악과 MV를 활용한다면 얼마나 큰 폭력을 일으킬 수 있을지 상상해...,2,1,중립,긍정,0.260518,0.657599,0.069187,0.005812,0.006884,False,2->1,6
10,UgwXz4GxeUUdx4towX54AaABAg,GROUP000061,1wGOHbcQKIc,이 노래가 헐스 리벤트리 출간되었고 이미 잘 알려진 노래입니다.,이 노래가 헐스 리벤트리 출간되었고 이미 잘 알려진 노래입니다.,2,1,중립,긍정,0.108621,0.857143,0.028847,0.002519,0.002870,False,2->1,6
13,Ugw1ISogEwX1crv0nGV4AaABAg,GROUP000068,1wGOHbcQKIc,노래와 MV가 이렇게 좋은가요?,노래와 MV가 이렇게 좋은가요?,1,2,긍정,중립,0.293696,0.205279,0.490677,0.004926,0.005422,False,1->2,3
14,Ugw8CgPsqJooXxTruOt4AaABAg,GROUP000083,1wGOHbcQKIc,노래가 좋고 춤도 좋죠,노래가 좋고 춤도 좋죠,1,0,긍정,부정,0.424486,0.296694,0.264147,0.007120,0.007554,False,1->0,3
17,UgygJeZpdOGmK3HvBSB4AaABAg,GROUP000090,1wGOHbcQKIc,2thumb up🔥,2thumb up🔥,1,0,긍정,부정,0.424486,0.296694,0.264147,0.007120,0.007554,False,1->0,2
18,UgwqSUDqUtCev_qyDw14AaABAg,GROUP000094,1wGOHbcQKIc,눈을 뜨라,눈을 뜨라,2,0,중립,부정,0.424486,0.296694,0.264147,0.007120,0.007554,False,2->0,1
23,UgwgHPz-QcR29lgluJd4AaABAg,GROUP000112,1wGOHbcQKIc,솔직히 메타버스 그룹이라는 콘셉트는 전혀 마음에 안 들지만 노래는 중독성 있어.,솔직히 메타버스 그룹이라는 콘셉트는 전혀 마음에 안 들지만 노래는 중독성 있어.,1,3,긍정,혼합,0.042264,0.242892,0.300178,0.395758,0.018907,False,1->3,11
26,Ugx2Svgk4xouACKImmN4AaABAg,GROUP000125,1wGOHbcQKIc,"오, K/DA 짝퉁이네. 다들 듣고도 모르겠어? ㅋㅋ 이 여자들의 목소리에 재능이 ...","오, K/DA 짝퉁이네. 다들 듣고도 모르겠어? ㅋㅋ 이 여자들의 목소리에 재능이 ...",0,1,부정,긍정,0.083427,0.706323,0.185961,0.007906,0.016383,False,0->1,21


검증 오류 수: 19


## 9 모델이 이용한 특징
LR/SVM은 클래스별 계수, NB는 클래스별 로그확률, XGBoost는 전체 중요도입니다. 서로 같은 의미의 가중치가 아닙니다. LSTM의 어휘 계수 표는 제공하지 않습니다.

In [20]:
weights = feature_weights(model)
if weights.empty:
    print("선택 모델은 어휘별 선형 계수를 제공하지 않습니다.")
else:
    display(weights.groupby("class_label", group_keys=False).tail(8))

,class_label,class_name,feature,weight,weight_type
4954,all,all,없어,0.027790,tree_importance
4955,all,all,이제,0.029466,tree_importance
4956,all,all,모두,0.029509,tree_importance
4957,all,all,-,0.029988,tree_importance
4958,all,all,(,0.039851,tree_importance
4959,all,all,봐,0.042492,tree_importance
4960,all,all,않은,0.043866,tree_importance
4961,all,all,같은,0.055642,tree_importance


## 10 검증 결과 저장
같은 설정으로 스크립트 흐름을 실행합니다. 다시 학습하지만 같은 데이터/seed/설정의 재현성 확인이며 새로운 평가 자료가 아닙니다.

In [21]:
output, saved_scores, metadata = run_experiment(CSV_PATH, plots=True,
    model_names=MODEL_NAMES, include_lstm=INCLUDE_LSTM, lstm_epochs=LSTM_EPOCHS)
print("저장:", output)
assert metadata["test_evaluated"] is False
assert not list(output.glob("test_*"))

저장: C:\Users\ktc system\나\3-2\culture_NLP\sub_assignment\261008\CNT3054-main\emotion_analysis\outputs\run_20261011T100806995282Z


## 11 개선 가설 기록
GT 기준, 판단 대상, 번역으로 달라진 의미, 반어, 혼합과 판단 불가의 혼동을 검증 오류에서 점검하세요. 후보/전처리/설정을 고정한 뒤 test를 평가합니다. 교안 examples는 별도 0/1 합성 과제입니다.

## 12 마지막 테스트
기본 RUN_FINAL_TEST=False입니다. test 결과를 보고 모델이나 seed를 바꾸면 새 독립 평가 자료가 필요합니다.

In [24]:
RUN_FINAL_TEST = True
if RUN_FINAL_TEST:
    final_output, _, final_metadata = run_experiment(CSV_PATH, evaluate_test=True,
        plots=True, model_names=MODEL_NAMES, include_lstm=INCLUDE_LSTM,
        lstm_epochs=LSTM_EPOCHS)
    display(pd.DataFrame(final_metadata["test_scores"]))
    print("최종 결과:", final_output)
else:
    print("테스트 미평가")

,model,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
0,majority_baseline,0.450,0.09000,0.200000,0.124138,0.279310
1,xgboost,0.475,0.31619,0.313889,0.310303,0.471515


최종 결과: C:\Users\ktc system\나\3-2\culture_NLP\sub_assignment\261008\CNT3054-main\emotion_analysis\outputs\run_20261011T101035163268Z


## 13 보고서에 기록할 내용
사용 데이터와 0~4 라벨 기준, 실제 분할 수와 그룹 정책, 기준선·후보 validation 점수, 선택 이유, 자체 오류 1개와 한계를 기록합니다. scores와 오류가 만들어지기 전에는 성능을 적지 않습니다.